In [7]:
import xml.etree.ElementTree as ET
import numpy as np
import glob
import os.path
import matplotlib.pyplot as plt
import scipy.interpolate
import re
import pdb
from kraken.lib import segmentation
from scipy.ndimage import gaussian_filter
from PIL import Image, ImageOps
import cv2
from skimage.filters import sobel

In [14]:
TOP_FRAC = 0.77
BOT_FRAC = 0.23

DIRNAME = "../val_data2/"
#DIRNAME = "staging/"
#DIRNAME = "./"
#DIRNAME = "modified/"

In [15]:
def get_line_spacing(baselines):
    #print(baselines)
    center_x = np.median([(l.T[0][0] + l.T[0][-1])/2 for l in baselines])
    
    center_ys = []
    for line in baselines:
        xs, ys = line.T
        center_ys.append(np.interp(center_x, xs, ys))

    med_spacing = np.median(np.diff(center_ys))
    return int(round(med_spacing))

def get_namespace(element):
    #rint(element.tag)
    m = re.match(r'\{.*\}', element.tag)
    return m.group(0)[1:-1] if m else ''    

In [16]:
def subtract_bkd(image):
    bkd = np.percentile(image, 80, axis=0)
    cols = np.arange(image.shape[1])
    coeffs = np.polyfit(cols, bkd, 2)
    smoothed_bkd = np.polyval(coeffs, cols)
    new_image = np.copy(image) / smoothed_bkd * np.median(smoothed_bkd)
    return new_image

def extract_line_image(image, output_filename, baseline, polygon, final_height=128):
    #f = open("top_bot_dists.txt", "a") 
    
    mask = np.zeros([image.size[1], image.size[0]], dtype=np.uint8)
    #pdb.set_trace()
    cv2.drawContours(mask, [np.array(polygon)], -1, (255), thickness=cv2.FILLED)
    
    baseline_dense_xs = np.arange(image.size[0])
    baseline_dense_ys = np.interp(baseline_dense_xs, baseline[:,0], baseline[:,1])

    start = None
    end = None
    
    top_dists = []
    bot_dists = []
    heights = []
    
    for c in range(mask.shape[1]):
        if np.sum(mask[:,c]) == 0: continue
        if start is None:
            start = c
        filled = np.nonzero(mask[:,c])[0]
        top_dists.append(baseline_dense_ys[c] - min(filled))
        bot_dists.append(max(filled) - baseline_dense_ys[c])
        heights.append(max(filled) - min(filled) + 1)
        end = c
    
    #1.23 and 0.97 were derived from analyzing all the top and bottom distances across all the files
    master_height_from_tops = np.percentile(top_dists, 80) / TOP_FRAC / 1.23
    master_height_from_bot = np.percentile(bot_dists, 80) / BOT_FRAC / 0.97

    master_height = int(round(np.percentile(heights, 80)))

    #f.write("{} {} {}\n".format(master_height_from_tops, master_height_from_bot, master_height))
    
    if master_height > 2 * min(master_height_from_tops, master_height_from_bot):
        print("Triggering safeguard", output_filename)
        master_height = 1.2 * min(master_height_from_tops, master_height_from_bot) 
    
    master_top_dist = int(round(TOP_FRAC * master_height))
    master_bot_dist = int(round(BOT_FRAC * master_height))
    
    final_image = np.zeros((master_top_dist + master_bot_dist, end - start))
    image_arr = np.copy(np.array(image))
    #image_arr[mask == 0] = np.percentile(image_arr[mask != 0], 90)
    
    for c in range(start, end):
        rel_positions_orig = np.arange(mask.shape[0]) - baseline_dense_ys[c]
        rel_positions_target = np.arange(final_image.shape[0]) - master_top_dist
        final_image[:,c - start] = np.interp(rel_positions_target, rel_positions_orig, image_arr[:,c])

    final_image = subtract_bkd(final_image)
    final_image = (np.median(final_image) - final_image) / (np.percentile(final_image, 90) - np.percentile(final_image, 10))
    final_image = 255 * (final_image - final_image.min()) / (final_image.max() - final_image.min())
    #final_image[master_top_dist] = 0
    pil_image = Image.fromarray(final_image)
    final_width = round(final_height / final_image.shape[0] * final_image.shape[1])
    pil_image = pil_image.resize((final_width, final_height)).convert("RGB")
    pil_image.save(output_filename)

    if np.random.randint(100) == 0:
        plt.figure()
        plt.imshow(pil_image)

    #f.close()
                

for filename in glob.glob(DIRNAME + "*.xml"):
    baselines = []
    print(filename)
    tree = ET.parse(filename)
    root = tree.getroot()
    ns = {"ns": get_namespace(tree.getroot())}
    ET.register_namespace('', ns['ns'])

    image_filename = root.find('ns:Page', ns).get('imageFilename')
    image = Image.open(DIRNAME + image_filename).convert('L')
    
    for text_region in root.findall('.//ns:TextRegion', ns):
        for lineno, text_line in enumerate(text_region.findall('.//ns:TextLine', ns)):
            if "margin" in (text_line.get("custom") or ""): continue
            baseline = text_line.find('ns:Baseline', ns).get('points')
            baselines.append(np.array([p.split(",") for p in baseline.split(" ")], dtype=int))

    polygons = segmentation.calculate_polygonal_environment(image, baselines=baselines, scale=(0, 1800))
    
    for i in range(len(baselines)):
        line_im_filename = "line_{}_{}".format(i, image_filename)
        line_im_filename, _ = os.path.splitext(line_im_filename)
        line_im_filename += ".png"
        extract_line_image(image, DIRNAME + line_im_filename, baselines[i], polygons[i]) 

../val_data2/IMG_0018B.xml
../val_data2/Copy of Phillipps 39F.xml
../val_data2/Copy of Phillipps 5F.xml
../val_data2/Copy of Phillipps 4F.xml
../val_data2/185r.xml
../val_data2/187r.xml
../val_data2/Copy of Phillipps 9F.xml
../val_data2/124r.xml
../val_data2/York_39.pdf_page_1.xml
../val_data2/7_d20e5_default.xml
../val_data2/187v.xml


In [5]:
np.array(image).shape

(778, 1023)

In [6]:
image.size[0]

1023